# Compare Prediction with validation data

Plotted on this [map](https://terriamap.p.niva.no/#start=%7B%22version%22%3A%228.0.0%22%2C%22initSources%22%3A%5B%7B%22stratum%22%3A%22user%22%2C%22models%22%3A%7B%22%2F%2FMarint+Natur+Kart%22%3A%7B%22isOpen%22%3Atrue%2C%22knownContainerUniqueIds%22%3A%5B%22%2F%22%5D%2C%22type%22%3A%22group%22%7D%2C%22no.niva.nkm%3Anisjedata-substrat-xgbclassifier_norge%22%3A%7B%22knownContainerUniqueIds%22%3A%5B%22%2F%2FMarint+Natur+Kart%22%5D%2C%22type%22%3A%22wfs%22%7D%2C%22no.niva.nkm%3Aaquamonitor_blotbunn_stations%22%3A%7B%22knownContainerUniqueIds%22%3A%5B%22%2F%2FBunntyper+Validering%22%5D%2C%22type%22%3A%22wfs%22%7D%2C%22no.niva.nkm%3Aaquamonitor_hardbunn_stations%22%3A%7B%22knownContainerUniqueIds%22%3A%5B%22%2F%2FBunntyper+Validering%22%5D%2C%22type%22%3A%22wfs%22%7D%2C%22%2F%22%3A%7B%22type%22%3A%22group%22%7D%2C%22%2F%2FBunntyper+Validering%22%3A%7B%22knownContainerUniqueIds%22%3A%5B%22%2F%22%5D%2C%22type%22%3A%22group%22%7D%7D%2C%22workbench%22%3A%5B%22no.niva.nkm%3Aaquamonitor_hardbunn_stations%22%2C%22no.niva.nkm%3Aaquamonitor_blotbunn_stations%22%2C%22no.niva.nkm%3Anisjedata-substrat-xgbclassifier_norge%22%5D%2C%22timeline%22%3A%5B%22no.niva.nkm%3Aaquamonitor_hardbunn_stations%22%2C%22no.niva.nkm%3Aaquamonitor_blotbunn_stations%22%2C%22no.niva.nkm%3Anisjedata-substrat-xgbclassifier_norge%22%5D%2C%22initialCamera%22%3A%7B%22west%22%3A12.643203735351564%2C%22south%22%3A66.61130843570749%2C%22east%22%3A13.923797607421877%2C%22north%22%3A66.88104124658943%7D%2C%22homeCamera%22%3A%7B%22west%22%3A4%2C%22south%22%3A57.00000000000001%2C%22east%22%3A32%2C%22north%22%3A72%7D%2C%22viewerMode%22%3A%222d%22%2C%22showSplitter%22%3Afalse%2C%22splitPosition%22%3A0.5%2C%22settings%22%3A%7B%22baseMaximumScreenSpaceError%22%3A2%2C%22useNativeResolution%22%3Afalse%2C%22alwaysShowTimeline%22%3Afalse%2C%22baseMapId%22%3A%22basemap-openstreetmap%22%2C%22terrainSplitDirection%22%3A0%2C%22depthTestAgainstTerrainEnabled%22%3Afalse%7D%2C%22stories%22%3A%5B%5D%7D%5D%7D)

In [1]:
import geopandas as gpd
import pandas as pd
from shapely import wkt

In [2]:
def summarize_points_in_bunn_types(points_gdf, polygons_gdf):
    total_points = len(points_gdf)

    rows = []
    for bunn_type in polygons_gdf["BunnType"].unique():
        sel_polygons = polygons_gdf[polygons_gdf["BunnType"] == bunn_type][["BunnType", "geometry"]]

        pts_in = gpd.sjoin(points_gdf, sel_polygons, how="inner", predicate="within")
        n_in = len(pts_in)

        # points correctly inside this bunn_type among all points
        pct_of_all = n_in / total_points * 100 if total_points > 0 else 0
        rows.append(
            {
                "BunnType": bunn_type,
                "n_in_type": n_in,
                "pct": pct_of_all,
            }
        )

    # points in any polygon
    any_poly = gpd.sjoin(points_gdf, polygons_gdf[["BunnType", "geometry"]], how="inner", predicate="within")
    n_in_any = len(any_poly)
    n_outside = total_points - n_in_any

    df = pd.DataFrame(rows)
    df["total_points"] = total_points
    df["n_in_any_polygon"] = n_in_any
    df["n_outside_all_polygons"] = n_outside

    df["pct_adj_for_outside"] = df["n_in_type"] / n_in_any * 100 if n_in_any > 0 else 0

    return df

In [3]:
gdf_predict = gpd.read_parquet(
    "gs://niva-geodata/MarintNaturKart/results/nisjedata-substrat-xgbclassifier_norge_latest_25833.geo.parquet"
)

In [4]:
typename = "no.niva.nkm:aquamonitor_blotbunn_stations"
wfs_url_template = (
    "https://terriamap.p.niva.no/geoserver/no.niva.nkm/ows"
    "?service=WFS&version=1.0.0&request=GetFeature"
    "&typeName={}"
    "&outputFormat=application/json"
)

In [5]:
gdf_blotbunn = (
    gpd.read_file(wfs_url_template.format(typename))
    .to_crs(gdf_predict.crs)
    .drop_duplicates(subset=["geometry"])
    .reset_index(drop=True)
)

In [6]:
summarize_points_in_bunn_types(gdf_blotbunn, gdf_predict)

,BunnType,n_in_type,pct,total_points,n_in_any_polygon,n_outside_all_polygons,pct_adj_for_outside
0,fastbunn,196,13.378840,1465,1386,79,14.141414
1,blanding,19,1.296928,1465,1386,79,1.370851
2,løsbunn,1171,79.931741,1465,1386,79,84.487734


In [7]:
typename = "no.niva.nkm:aquamonitor_hardbunn_stations"
gdf_hardbunn = (
    gpd.read_file(wfs_url_template.format(typename))
    .to_crs(gdf_predict.crs)
    .drop_duplicates(subset=["geometry"])
    .reset_index(drop=True)
)

In [8]:
summarize_points_in_bunn_types(gdf_hardbunn, gdf_predict)

,BunnType,n_in_type,pct,total_points,n_in_any_polygon,n_outside_all_polygons,pct_adj_for_outside
0,fastbunn,91,33.579336,271,145,126,62.758621
1,blanding,6,2.214022,271,145,126,4.137931
2,løsbunn,48,17.712177,271,145,126,33.103448
